# Reinforce
First we look for th best combination possible of learning rate (just for the actorb bc we're testing the reinforce algorithm), and value of gamma.

- GAMMA:
    - 0.99
    - 0.995
    - 0.999

- learning rate
    - 1e-5
    - 5e-5
    - 1e-4
    - 5e-4 

In [1]:
import os
os.environ["WANDB_MODE"] = "offline"
import random
import gymnasium as gym
import torch
import numpy as np
import wandb

from agent import Agent, Policy

SEED = 42

def main():
    os.makedirs("part1/plots/hp_tuning", exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Training on device:", device)

    n_episodes = 20000
    alg = "REINFORCE"
    baseline = True

    actor_lr_values = [1e-5, 5e-5, 1e-4, 5e-4]
    gamma_values = [0.99, 0.995, 0.999]
    run_id = 0

    for learning_rate in actor_lr_values:
        for gamma in gamma_values:
            
            # Set seeds to ensure reproducibility for each run
            current_seed = SEED + run_id
            run_id += 1
            np.random.seed(current_seed)
            torch.manual_seed(current_seed)
            random.seed(current_seed)
            torch.cuda.manual_seed_all(current_seed)
            
            # print configuration
            print(f"\n\nTuning REINFORCE \n\tlearning rate: {learning_rate}  \n\tgamma: {gamma}")

            # setting up wandb config 
            wandb.init(
                project="FAIML-RL-26-hp_tuning", 
                name=f"LR_{learning_rate}_GAMMA_{gamma}", 
                config={
                    "algorithm": alg,
                    "baseline": baseline,
                    "actor_lr": learning_rate,
                    "gamma": gamma,
                    "n_episodes": n_episodes,
                })

            env = gym.make('Hopper-v4')
            
            dim_state_space = env.observation_space.shape[0]
            dim_action_space = env.action_space.shape[0]

            # agent and policy initialization
            policy = Policy(dim_state_space, dim_action_space).to(device)
            agent = Agent(policy, device=device, actor_lr=learning_rate, gamma=gamma)

            n_steps_tot = 0

            for ep in range(n_episodes):  
                done = False
                state, info = env.reset(seed=current_seed + ep)  # Reset environment to initial state
                ep_reward = 0.0
                n_steps_inside_episode = 0

                while not done:  
                    action, action_log_probs = agent.get_action(state)  # Sample random action
                    action_numpy = action.cpu().detach().numpy()

                    next_state, reward, terminated, truncated, _ = env.step(action_numpy)  # Step the simulator to the next timestep
                    done = terminated or truncated

                    agent.store_outcome(state, next_state, action_log_probs, reward, done)  

                    # updates
                    state = next_state
                    ep_reward += reward
                    n_steps_tot += 1
                    n_steps_inside_episode += 1

                loss, _ = agent.update_policy(baseline=baseline, algorithm=alg)  

                # log all results to wandb
                wandb.log({
                    "Episode reward": ep_reward,
                    "n_steps": n_steps_tot,
                    "n_steps_inside_episode": n_steps_inside_episode,
                    "loss": loss,
                }) 

                if ep % 500 == 0:
                    print(f"Episode: {ep}, Reward: {ep_reward}")

            env.close()
            wandb.finish()

if __name__ == '__main__':
    main()

Training on device: cpu


Tuning REINFORCE 
	learning rate: 1e-05  
	gamma: 0.99


/Users/riymchaouiaziz/Desktop/FAIDML-project/FAIML-RL-26/FAIMDL/lib/python3.10/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Episode: 0, Reward: 19.37948393671164
Episode: 500, Reward: 9.624333599911264
Episode: 1000, Reward: 5.858475390521438
Episode: 1500, Reward: 10.229875625733547
Episode: 2000, Reward: 11.464643020702267
Episode: 2500, Reward: 13.862122237470063
Episode: 3000, Reward: 18.011497579861107
Episode: 3500, Reward: 14.317238735337245
Episode: 4000, Reward: 88.45571568888633
Episode: 4500, Reward: 131.28930304240004
Episode: 5000, Reward: 202.86525574693184
Episode: 5500, Reward: 161.98835851655883
Episode: 6000, Reward: 169.2615178365833
Episode: 6500, Reward: 217.74115828349406
Episode: 7000, Reward: 119.55139330178316
Episode: 7500, Reward: 163.76708282091693
Episode: 8000, Reward: 61.8370646261472
Episode: 8500, Reward: 202.83384145886652
Episode: 9000, Reward: 179.3310424852985
Episode: 9500, Reward: 201.3548616867682
Episode: 10000, Reward: 185.11463509346657
Episode: 10500, Reward: 194.53674078288915
Episode: 11000, Reward: 188.45064072407152
Episode: 11500, Reward: 193.11240933507494
E

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Episode reward,▁▂▁▁▁▁▂▂▇▃▆▅▇▃█▇▂▇▆█▇▇▅▇█▇▇▇▇▇▇▃▇▇▂▇▇▇▇▆
loss,▁▁▂▂▁▂▁▂▂▁▅█▅▆▅▇▇▇▁▇▇▄█▅▇▃█▇██▇▇▇▄▇█▆▇▇█
n_steps,▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇██
n_steps_inside_episode,▁▂▁▂▃▃▃▇▇█▃▆▇▆▆▄▆█▆▂█▆▇▇▇▇█▆▆▇▆▆▇▆▆▆▇█▇▇
Episode reward,192.76387
loss,249.85909
n_steps,1470436
n_steps_inside_episode,95


...



Tuning REINFORCE 
	learning rate: 0.0005  
	gamma: 0.999


/Users/riymchaouiaziz/Desktop/FAIDML-project/FAIML-RL-26/FAIMDL/lib/python3.10/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Episode: 0, Reward: 12.99044894392947
Episode: 500, Reward: 175.59610970317215
Episode: 1000, Reward: 190.75432214464115
Episode: 1500, Reward: 167.49088617995264
Episode: 2000, Reward: 137.49367324600183
Episode: 2500, Reward: 149.35043021857928
Episode: 3000, Reward: 179.50223239825985
Episode: 3500, Reward: 203.05871847614938
Episode: 4000, Reward: 228.75765567534583
Episode: 4500, Reward: 235.68413254103797
Episode: 5000, Reward: 375.9256158269811
Episode: 5500, Reward: 561.3386074622707
Episode: 6000, Reward: 1025.9866571427563
Episode: 6500, Reward: 1027.4280311388113
Episode: 7000, Reward: 865.6053283187125
Episode: 7500, Reward: 1026.7642672670058
Episode: 8000, Reward: 404.5723179649806
Episode: 8500, Reward: 273.938578406549
Episode: 9000, Reward: 443.7914727676628
Episode: 9500, Reward: 497.3426966437838
Episode: 10000, Reward: 408.81464421173416
Episode: 10500, Reward: 150.98823011754857
Episode: 11000, Reward: 343.77349232982044
Episode: 11500, Reward: 283.4260188361328
Ep

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Episode reward,▁▂▁▁▁▁▁▁▂▂▆███▅▁▆▃▂▃▃▃▄▄▃▂▂▁▂▂▂▂▁▁▁▂▂▃██
loss,▃▂▂▃▂▄▄███▄▄▅▄▆▄▄▄▃▃▃▃▃▃▃▃▂▂▃▂▃▂▂▂▂▂▂▃▁▇
n_steps,▁▁▁▁▁▂▂▃▃▃▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇██
n_steps_inside_episode,▁▁▂▁▁▂▂▂▂▃▄▃▃▂▂▂▂▂▃▃▄▂▂▂▂▂▂▂▄▂▂▂▂▂▂▂▂▄██
Episode reward,964.76757
loss,1364.47974
n_steps,5557570
n_steps_inside_episode,1000


# Actor-Critic

In [2]:
import os
os.environ["WANDB_MODE"] = "offline"
import random
import gymnasium as gym
import torch
import numpy as np
import wandb

from agent import Agent, Policy

SEED = 42

def main():
    #os.makedirs("part1/plots/hp_tuning", exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Training on device:", device)

    n_episodes = 20000
    alg = "Actor-Critic"
    baseline = False

    gamma = 0.999 # chosen from previous HP
    actor_learning_rate = 1e-4 # chosen from previous HP

    critic_lr_values = [1e-4, 5e-4, 1e-3, 3e-3]
    run_id = 0

    for learning_rate in critic_lr_values:
            
        # Set seeds to ensure reproducibility for each run
        current_seed = SEED + run_id
        run_id += 1
        np.random.seed(current_seed)
        torch.manual_seed(current_seed)
        random.seed(current_seed)
        torch.cuda.manual_seed_all(current_seed)
        
        # print configuration
        print(f"\n\nTuning Actor-Critic \n\tlearning rate: {learning_rate}")

        # setting up wandb config 
        wandb.init(
            project="FAIML-RL-26-hp_tuning", 
            name=f"LR_critic_{learning_rate}", 
            config={
                "algorithm": alg,
                "baseline": baseline,
                "actor_lr": actor_learning_rate,
                "critic_lr": learning_rate,
                "gamma": gamma,
                "n_episodes": n_episodes,
            })

        env = gym.make('Hopper-v4')
        
        dim_state_space = env.observation_space.shape[0]
        dim_action_space = env.action_space.shape[0]

        # agent and policy initialization
        policy = Policy(dim_state_space, dim_action_space).to(device)
        agent = Agent(policy, device=device, actor_lr=actor_learning_rate, critic_lr=learning_rate, gamma=gamma)

        n_steps_tot = 0

        for ep in range(n_episodes):  
            done = False
            state, info = env.reset(seed=current_seed + ep)  # Reset environment to initial state
            ep_reward = 0.0
            n_steps_inside_episode = 0

            while not done:  
                action, action_log_probs = agent.get_action(state)  # Sample random action
                action_numpy = action.cpu().detach().numpy()

                next_state, reward, terminated, truncated, _ = env.step(action_numpy)  # Step the simulator to the next timestep
                done = terminated or truncated

                agent.store_outcome(state, next_state, action_log_probs, reward, done)  

                # updates
                state = next_state
                ep_reward += reward
                n_steps_tot += 1
                n_steps_inside_episode += 1

            actor_loss, critic_loss = agent.update_policy(baseline=baseline, algorithm=alg)  

            # log all results to wandb
            wandb.log({
                    "Episode number": ep,
                    "Episode reward": ep_reward,
                    "n_steps": n_steps_tot,
                    "n_steps_inside_episode": n_steps_inside_episode,
                    "actor_loss": actor_loss,
                    "critic_loss": critic_loss,}) 

            if ep % 500 == 0:
                print(f"Episode: {ep}, Reward: {ep_reward:}")

        env.close()
        wandb.finish()

if __name__ == '__main__':
    main()

Training on device: cpu


Tuning Actor-Critic 
	learning rate: 0.0001


/Users/riymchaouiaziz/Desktop/FAIDML-project/FAIML-RL-26/FAIMDL/lib/python3.10/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Episode: 0, Reward: 19.37948393671164
Episode: 500, Reward: 7.040236102283682
Episode: 1000, Reward: 9.674131121564121
Episode: 1500, Reward: 28.937878075400377
Episode: 2000, Reward: 223.08607099168495
Episode: 2500, Reward: 209.54931623569445
Episode: 3000, Reward: 209.07680225267652
Episode: 3500, Reward: 215.73207221383976
Episode: 4000, Reward: 225.01406888202146
Episode: 4500, Reward: 223.56902972208528
Episode: 5000, Reward: 226.04347949962101
Episode: 5500, Reward: 226.88077724722572
Episode: 6000, Reward: 224.3684157633188
Episode: 6500, Reward: 201.39303456046247
Episode: 7000, Reward: 198.3682841850081
Episode: 7500, Reward: 194.7335527254085
Episode: 8000, Reward: 189.44113397903692
Episode: 8500, Reward: 208.9918000368985
Episode: 9000, Reward: 202.5960079632577
Episode: 9500, Reward: 211.51525456144185
Episode: 10000, Reward: 186.63006820671154
Episode: 10500, Reward: 179.22795577819744
Episode: 11000, Reward: 172.26630955287357
Episode: 11500, Reward: 174.91633072443622


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Episode number,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
Episode reward,▁▁▁▁▅▅▅▆▆▅▅▂▅▅▆▅▅▅▅▅▅▅▅▅▅▅▅▄▅▅▅▅▆▅▅▅▇▆▅█
actor_loss,▁▃▂▆▇█▇▇▆▇▇▇▅▇▅▆▅▆▆▆▅▁▅▅▅▄▄▄▅▂▄▅▄▃▅▄▃▃▂▂
critic_loss,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▅▄▅▆▆▆▆▇▇▇▇▇████▆
n_steps,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇██
n_steps_inside_episode,▁▁▂▃█▅▆▇▆▆▆▆▆▆▆▄▄▆▃▆▆▆▆▆▇▆▆▃▆▆▅▆▃▆▆▆▆▆▇▇
Episode number,19999
Episode reward,246.71627
actor_loss,4.94528
critic_loss,156.00111
n_steps,1803368


...



Tuning Actor-Critic 
	learning rate: 0.003


/Users/riymchaouiaziz/Desktop/FAIDML-project/FAIML-RL-26/FAIMDL/lib/python3.10/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Episode: 0, Reward: 28.54809469044944
Episode: 500, Reward: 190.15725744941918
Episode: 1000, Reward: 250.145454362953
Episode: 1500, Reward: 336.64054693744544
Episode: 2000, Reward: 670.6968951765474
Episode: 2500, Reward: 414.71817783606656
Episode: 3000, Reward: 1030.1449319869603
Episode: 3500, Reward: 1033.568899339398
Episode: 4000, Reward: 681.1718928559754
Episode: 4500, Reward: 1005.8193384949599
Episode: 5000, Reward: 1022.0575336791237
Episode: 5500, Reward: 1017.4720668846127
Episode: 6000, Reward: 1024.35373634309
Episode: 6500, Reward: 1011.7499278682891
Episode: 7000, Reward: 998.9599649457763
Episode: 7500, Reward: 998.220886280931
Episode: 8000, Reward: 988.4007203977559
Episode: 8500, Reward: 1002.7807621840907
Episode: 9000, Reward: 992.1639146588938
Episode: 9500, Reward: 983.4262679624454
Episode: 10000, Reward: 992.0163324616321
Episode: 10500, Reward: 993.5713470496416
Episode: 11000, Reward: 998.3374310821756
Episode: 11500, Reward: 992.402365240699
Episode: 12

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Episode number,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇███
Episode reward,▁▁▂▂▂▄▆▆▇█▆█████████████████████████████
actor_loss,▅██▄▃▆▂▃▂▃▃▃▂▁▁▂▂▂▂▂▂▂▂▁▂▂▂▂▃▂▂▂▁▂▂▁▂▂▂▂
critic_loss,▂▄▁▁▁▃▄▅█▇▇████▅▇██▇████████████████████
n_steps,▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇██
n_steps_inside_episode,▁▁▆█▇███████████████████████████████████
Episode number,19999
Episode reward,986.02729
actor_loss,0.0691
critic_loss,249.10738
n_steps,18104209
